In [1]:
#BSWC - Building Systems with ChatGPT API

#Process Inputs: Chain of Thought Reasoning

In [2]:
#1.SETTING UP THE OPENAI CLIENT

import os
# import → Keyword used to bring a library into the notebook
# os → Built-in library for interacting with the operating system (e.g., reading environment variables)

from dotenv import load_dotenv
# from → Keyword to import from a specific library
# dotenv → Library (python-dotenv) for reading key-value settings from a .env file
# import → Keyword used to bring in the specified item
# load_dotenv → Function that loads variables from a .env file into the environment

from openai import OpenAI
# from → Keyword to import from a specific library
# openai → Library used to connect to OpenAI's models
# import → Keyword used to bring in the specified item
# OpenAI → Class used to create a client that sends requests to OpenAI's API

load_dotenv(dotenv_path=".env")
# load_dotenv → Function that loads variables from a .env file into the environment
# (dotenv_path=".env") → Argument specifying the path/name of the file to read

openai_api_key = os.getenv("OPENAI_API_KEY")
# openai_api_key → Variable name storing the result
# = → Assignment operator
# os → Built-in operating system library
# .getenv → Function that reads the value of an environment variable
# ("OPENAI_API_KEY") → Argument specifying the name of the variable to read (your secret API key)

client = OpenAI()
# client → Variable name storing the result
# = → Assignment operator
# OpenAI() → Creates the API client object (automatically uses the API key from the environment)

# This cell loads the API key from a .env file and creates the OpenAI client object used by later cells to send requests to the model.

In [3]:
#2.FUNCTION TO GET A COMPLETION FROM A LIST OF MESSAGES

def get_completion_from_messages(messages,
                                  model="gpt-5.6-luna",
                                  temperature=1, max_tokens=500):
# def → Keyword used to define a function
# get_completion_from_messages → Name given to the function
# (messages, model="gpt-5.6-luna", temperature=1, max_tokens=500) → Parameters: the conversation list, and three settings with default values
# messages → Parameter holding the full list of conversation messages (no default, must be provided)
# model="gpt-5.6-luna" → Parameter setting which model to use, defaulting to gpt-5.6-luna
# temperature=1 → Parameter controlling randomness, defaulting to 1 (gpt-5.6-luna's only supported value)
# max_tokens=500 → Parameter limiting the length of the reply, defaulting to 500 tokens
# : → Marks the start of the function body

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=max_tokens,
    )
    # response → Variable name storing the result
    # = → Assignment operator
    # client → The OpenAI client object created earlier
    # .chat.completions → Attribute giving access to the Chat Completions API
    # .create → Function that sends a request to the model and gets a reply
    # ( → Opens the function's arguments
    # model=model → Argument passing the model name given to the function
    # messages=messages, → Argument passing the conversation list given to the function
    # temperature=temperature, → Argument passing the randomness setting given to the function
    # max_completion_tokens=max_tokens, → Argument passing the output length limit, using the parameter name required by gpt-5.6-luna
    # ) → Closes the function's arguments

    return response.choices[0].message.content
    # return → Keyword that sends a value back from the function
    # response → The reply object returned by the model
    # .choices → Attribute holding a list of possible responses (usually just one)
    # [0] → Indexing operator selecting the first (and only) response
    # .message → Attribute holding the message object of that response
    # .content → Attribute holding the actual answer text

# This function accepts a full messages list (for multi-turn conversations) and lets you customize the model, temperature, and max_tokens, 
# returning just the model's reply text. It uses max_completion_tokens, the parameter name required by gpt-5.6-luna.

In [4]:
#3.CHAIN-OF-THOUGHT PROMPTING WITH DELIMITERS - PRODUCT QUERY REASONING

delimiter = "####"
# delimiter → Variable name storing the result
# = → Assignment operator
# "####" → String used as a separator to clearly mark the boundaries between the customer's query and different reasoning steps

system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}. 

Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product cateogry doesn't count. 

Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products: 
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99

Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.

Step 4:{delimiter}: If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information. 

Step 5:{delimiter}: First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""
# system_message → Variable name storing the result
# = → Assignment operator
# f""" ... """ → Triple-quoted f-string: a multi-line string that can insert variable values inside { }
# Follow these steps to answer the customer queries. → Overall instruction introducing a multi-step reasoning process
# The customer query will be delimited with four hashtags, i.e. {delimiter}. → Tells the model how the customer's input will be marked off 
#(inserts "####")

# Step 1:{delimiter} First decide whether the user is asking about a specific product... → First reasoning step: classify whether the question is 
#product-specific

# Step 2:{delimiter} If asking about specific products, identify whether they're in the list → Second reasoning step: match mentioned products 
#against the catalog

# All available products: → Label introducing the full product catalog
# 1. Product: TechPro Ultrabook ... Price: $799.99 → First catalog entry with its Category, Brand, Model Number, Warranty, Rating, Features, 
#Description, and Price

# 2. Product: BlueWave Gaming Laptop ... Price: $1199.99 → Second catalog entry, same structure
# 3. Product: PowerLite Convertible ... Price: $699.99 → Third catalog entry, same structure
# 4. Product: TechPro Desktop ... Price: $999.99 → Fourth catalog entry, same structure
# 5. Product: BlueWave Chromebook ... Price: $249.99 → Fifth catalog entry, same structure
# Step 3:{delimiter} ...list any assumptions that the user is making... → Third reasoning step: spot claims the customer assumed 
#(e.g. comparing sizes or warranties)

# Step 4:{delimiter}: ...figure out whether the assumption is true... → Fourth reasoning step: fact-check those assumptions against the real 
#catalog data

# Step 5:{delimiter}: First, politely correct the customer's incorrect assumptions... → Fifth reasoning step: instructs 
#how to respond — correct errors politely, stay within the 5 real products, and keep a friendly tone

# Use the following format: → Label introducing the exact output template
# Step 1:{delimiter} <step 1 reasoning> → Template line showing where Step 1's reasoning goes
# Step 2:{delimiter} <step 2 reasoning> → Template line showing where Step 2's reasoning goes
# Step 3:{delimiter} <step 3 reasoning> → Template line showing where Step 3's reasoning goes
# Step 4:{delimiter} <step 4 reasoning> → Template line showing where Step 4's reasoning goes
# Response to user:{delimiter} <response to customer> → Template line showing where the final customer-facing answer goes
# Make sure to include {delimiter} to separate every step. → Final rule reinforcing that every section must be split using "####"

# This cell builds a structured system prompt for a customer-support assistant: it forces the model to reason step-by-step 
# (classify the query, check products against a 5-item catalog, spot and verify customer assumptions, then respond) 
# with each step clearly separated by the #### delimiter, reducing hallucination and keeping answers grounded to real inventory.

In [5]:
#4.TESTING THE CHAIN-OF-THOUGHT SYSTEM WITH A CUSTOMER QUERY

user_message = f"""
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop"""
# user_message → Variable name storing the result
# = → Assignment operator
# f""" ... """ → Triple-quoted f-string: a multi-line string that can insert variable values inside { }
# by how much is the BlueWave Chromebook more expensive than the TechPro Desktop → The actual customer question, comparing prices of two 
#catalog products

# \ → Line continuation character: joins the two code lines into one continuous string with no line break in the text

messages = [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content': system_message} → Key-value pair inserting the full step-by-step reasoning prompt defined earlier
# {'role':'user' → Key-value pair marking this message as coming from the user

# 'content': f"{delimiter}{user_message}{delimiter}"} → Key-value pair wrapping the customer's question between #### delimiters, 
#exactly as the system prompt expects

response = get_completion_from_messages(messages)
# response → Variable name storing the result
# = → Assignment operator
# get_completion_from_messages → The function defined earlier that sends a full message list and returns the model's reply
# (messages) → Argument passing the conversation list defined above

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the model's step-by-step reasoning plus its final customer-facing answer

# This cell tests the chain-of-thought system: it asks a price-comparison question between the BlueWave Chromebook and TechPro Desktop, 
# wraps it in #### delimiters as required by the system prompt, sends it through get_completion_from_messages, and prints the full step-by-step 
# reasoning followed by the final response to the customer.

Step 1:#### The user is asking about two specific products: the BlueWave Chromebook and the TechPro Desktop.

Step 2:#### Both products are in the available product list. The BlueWave Chromebook costs $249.99, and the TechPro Desktop costs $999.99.

Step 3:#### The user assumes that the BlueWave Chromebook is more expensive than the TechPro Desktop.

Step 4:#### This assumption is incorrect. The BlueWave Chromebook is $750.00 cheaper than the TechPro Desktop ($999.99 − $249.99).

Response to user:#### The BlueWave Chromebook is not more expensive; it is **$750.00 cheaper** than the TechPro Desktop.


In [6]:
#5.TESTING THE CHAIN-OF-THOUGHT SYSTEM WITH AN OUT-OF-CATALOG QUESTION

user_message = f"""
do you sell tvs"""
# user_message → Variable name storing the result
# = → Assignment operator
# f""" ... """ → Triple-quoted f-string: a multi-line string that can insert variable values inside { }

# do you sell tvs → The actual customer question, asking about a product category not in the store's catalog 
#(TVs aren't among the 5 laptops/desktops listed)

messages = [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content': system_message} → Key-value pair inserting the full step-by-step reasoning prompt defined earlier
# {'role':'user' → Key-value pair marking this message as coming from the user

# 'content': f"{delimiter}{user_message}{delimiter}"} → Key-value pair wrapping the customer's question between #### delimiters, 
#exactly as the system prompt expects

response = get_completion_from_messages(messages)
# response → Variable name storing the result
# = → Assignment operator
# get_completion_from_messages → The function defined earlier that sends a full message list and returns the model's reply
# (messages) → Argument passing the conversation list defined above

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the model's step-by-step reasoning plus its final customer-facing answer

# This cell tests the chain-of-thought system with a question outside the store's product catalog (TVs), checking whether the model correctly 
# reasons through Step 1-2 (no matching product found) and responds by politely clarifying that only the 5 listed computer/laptop products are sold, 
# without inventing or hallucinating a TV product.

Step 1:#### The user is asking about a product category, not a specific product.  
Step 2:#### TVs are not among the five available products.  
Step 3:#### No assumptions were made.  
Step 4:#### There are no assumptions to verify.  
Response to user:#### We don’t currently sell TVs. We offer laptops and desktops, including the TechPro Ultrabook, BlueWave Gaming Laptop, PowerLite Convertible, TechPro Desktop, and BlueWave Chromebook.


In [1]:
#6.EXTRACTING JUST THE FINAL CUSTOMER-FACING RESPONSE (WITH ERROR HANDLING)

try:
# try → Keyword starting a block that may raise an error

    final_response = response.split(delimiter)[-1].strip()
    # final_response → Variable name storing the result
    # = → Assignment operator
    # response → The full model output, including all step-by-step reasoning plus the final answer
    # .split → Function that breaks a string into a list of pieces wherever the given separator appears
    # (delimiter) → Argument specifying "####" as the separator to split on
    # [-1] → Indexing operator selecting the LAST item in the resulting list (the text after the final ####, i.e. the actual customer response)
    # .strip() → Function that removes leading/trailing whitespace and newlines from that final piece

except Exception as e:
# except → Keyword catching an error if the try block fails
# Exception → The general error type being caught (catches any kind of error)
# as e → Stores the error details in a variable named e (not used further here, but available for debugging)
# : → Marks the start of the except-block

    final_response = "Sorry, I'm having trouble right now, please try asking another question."
    # final_response → Variable name storing the result
    # = → Assignment operator
    
    # "Sorry, I'm having trouble right now, please try asking another question." → Fallback message shown to the customer if 
    #extracting the response fails (e.g. the model didn't follow the #### format)

print(final_response)
# print → Built-in function that displays output
# (final_response) → Argument specifying what to display — either the extracted customer-facing answer or the fallback error message

# This cell extracts just the final answer meant for the customer, stripping away all the internal step-by-step reasoning: it splits the full 
# response by the #### delimiter and takes the last segment. If the model's output doesn't follow the expected format 
# (causing the split/extraction to fail), it falls back to a friendly error message instead of crashing or showing broken text.

Sorry, I'm having trouble right now, please try asking another question.
